# Warden NSFW image filtering

Local PyTorch inference for Falconsai/nsfw_image_detection. The model predicts normal or nsfw; Warden uses the NSFW probability to decide whether an image should be shown or hidden behind a warning.

In [1]:
# Run once if needed:
# %pip install torch transformers pillow

import os
# Work around duplicate Intel OpenMP runtimes in some Windows Conda environments.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

from pathlib import Path
import json
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

MODEL_PATH = Path('../models/nsfw-image-detection')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH.resolve()}')

processor = AutoImageProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForImageClassification.from_pretrained(
    MODEL_PATH, local_files_only=True, use_safetensors=False
).to(DEVICE)
model.eval()

print('device:', DEVICE)
print('labels:', model.config.id2label)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

device: cuda
labels: {0: 'normal', 1: 'nsfw'}


In [2]:
def classify_image(image_path: str, warning_threshold: float = 0.50) -> dict:
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f'Image not found: {path.resolve()}')

    image = Image.open(path).convert('RGB')
    inputs = processor(images=image, return_tensors='pt')
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

    with torch.inference_mode():
        probabilities = torch.softmax(model(**inputs).logits[0], dim=-1).cpu()

    scores = {
        model.config.id2label[index]: round(float(probabilities[index]), 4)
        for index in range(len(probabilities))
    }
    nsfw_score = scores.get('nsfw', 0.0)
    return {
        'image': str(path),
        'nsfw_score': nsfw_score,
        'categories': scores,
        'action': 'blur' if nsfw_score >= warning_threshold else 'show',
    }

In [12]:
# Replace this with a local test image.
result = classify_image('../data/images/image2.webp')
print(json.dumps(result, indent=2))

{
  "image": "..\\data\\images\\image2.webp",
  "nsfw_score": 0.0002,
  "categories": {
    "normal": 0.9998,
    "nsfw": 0.0002
  },
  "action": "show"
}


The 0.50 threshold is only a starting point. Calibrate it with safe and NSFW validation images before presenting it as a measured policy.